# NTX Q1 Stage 21 — Verified Execution Broker End-to-End Concurrency and Mandatory Mediation

Extends the 1,320-scenario broker study with concurrency, replay, stale-state, crash, expiry, and bypass tests. It is designed to demonstrate mandatory mediation rather than merely isolated broker correctness.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import pandas as pd, numpy as np, asyncio
STAGE="21_broker_e2e"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
REPO=ROOT/"repo"
SELFTEST=os.environ.get("NTX_SELFTEST","0")=="1"
if not SELFTEST and not REPO.exists():
    git_clone_frozen(REPO)
    run([sys.executable,"-m","pip","install","-q","-e",str(REPO/"niyamtrace")+"[dev,postgres]"])

In [ ]:
# First run the repository's security test suite and preserve raw output.
if not SELFTEST:
    tests=[
        REPO/"niyamtrace/tests/security/test_e2e_broker.py",
    ]
    test_rows=[]
    for t in tests:
        if t.exists():
            cp=run([sys.executable,"-m","pytest",str(t),"-q"],cwd=REPO/"niyamtrace",check=False,timeout=240)
            (OUT/(t.stem+".log")).write_text(cp.stdout,encoding="utf-8")
            test_rows.append({"test":str(t.relative_to(REPO)),"returncode":cp.returncode,"passed":cp.returncode==0})
        else:
            test_rows.append({"test":str(t),"returncode":None,"passed":False,"reason":"missing"})
    pd.DataFrame(test_rows).to_csv(OUT/"repository_security_tests.csv",index=False)
else:
    pd.DataFrame([{"test":"offline_selftest","returncode":0,"passed":True}]).to_csv(OUT/"repository_security_tests.csv",index=False)

## Stress-test input

The notebook accepts a case-level broker stress export in `NTX_BROKER_STRESS_RESULTS`. This keeps the stress generator independent from the metric code.

Required columns:

`case_id,scenario,expected_commit,actual_commit,latency_ms`

Recommended scenarios: `valid`, `replay`, `expired`, `stale_snapshot`, `actor_mismatch`, `tool_mismatch`, `arg_tamper`, `target_tamper`, `policy_change`, `concurrent_phantom`, `concurrent_write`, `crash_before_commit`, `direct_executor_bypass`.

In [ ]:
stress_file=os.environ.get("NTX_BROKER_STRESS_RESULTS","")
if stress_file and Path(stress_file).exists():
    x=pd.read_csv(stress_file)
    req={"case_id","scenario","expected_commit","actual_commit"}
    assert req.issubset(x.columns),f"Missing {req-set(x.columns)}"
    x["expected_commit"]=x.expected_commit.astype(bool)
    x["actual_commit"]=x.actual_commit.astype(bool)
    x["correct"]=x.expected_commit==x.actual_commit
    x["unsafe_commit"]=(~x.expected_commit)&x.actual_commit
    by=x.groupby("scenario").agg(n=("case_id","size"),accuracy=("correct","mean"),unsafe_commits=("unsafe_commit","sum")).reset_index()
    if "latency_ms" in x:
        lat=x.groupby("scenario").latency_ms.agg(["median",lambda s:s.quantile(.95)]).reset_index()
        lat.columns=["scenario","median_latency_ms","p95_latency_ms"]
        by=by.merge(lat,on="scenario",how="left")
    by.to_csv(OUT/"broker_scenario_metrics.csv",index=False)
    x.to_csv(OUT/"broker_case_results.csv",index=False)
    mandatory=("direct_executor_bypass" in set(x.scenario) and 
               int(x.loc[x.scenario=="direct_executor_bypass","unsafe_commit"].sum())==0)
    conc_n=int(x.scenario.astype(str).str.startswith("concurrent").sum())
    paper_ready=(len(x)>=1000 and mandatory and conc_n>=500 and int(x.unsafe_commit.sum())==0)
    write_json(OUT/"broker_e2e_summary.json",{
        "n":len(x),"accuracy":float(x.correct.mean()),"unsafe_commits":int(x.unsafe_commit.sum()),
        "mandatory_mediation_tested":bool(mandatory),"concurrency_cases":conc_n
    })
    record_status(OUT,STAGE,"PASS" if paper_ready else "PARTIAL",paper_ready=paper_ready)
    display(by)
else:
    print("Provide NTX_BROKER_STRESS_RESULTS after executing the concurrency harness.")

In [ ]:
z=package_stage(OUT,"NTX_Q1_21_BROKER_E2E_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)